# Deepfake Detection Beast Model - Training Notebook
### EfficientNet-B7 + Frequency Analysis + Attention Mechanism
**Datasets:** FaceForensics++, AI-Generated Images (Gemini/Grok), Custom Augmented

**Target:** >98% Accuracy, >0.99 AUC-ROC, State-of-the-art F1

---

## Phase 0: Environment Setup & GPU Check

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create project directories
import os
PROJECT_ROOT = '/content/drive/MyDrive/deepfake_beast'
os.makedirs(f'{PROJECT_ROOT}/datasets', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/datasets/ff++', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/datasets/ai_generated', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/logs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/exports', exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Install dependencies
!pip install -q timm albumentations facenet-pytorch opencv-python-headless \
    scikit-learn matplotlib seaborn tensorboard wandb \
    torch torchvision torchaudio --upgrade \
    grad-cam torchmetrics pillow tqdm gdown

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import timm
from facenet_pytorch import MTCNN
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.auto import tqdm
import json
import time
import warnings
warnings.filterwarnings('ignore')

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('WARNING: No GPU detected! Training will be extremely slow.')
    print('Go to Runtime > Change runtime type > GPU (T4 or better)')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

## Phase 1: Configuration

In [ ]:
# ============================================================
#  MASTER CONFIG - Tune everything from here
# ============================================================

CFG = {
    # --- Paths ---
    'project_root': PROJECT_ROOT,
    'ff_dataset_dir': f'{PROJECT_ROOT}/datasets/ff++',
    'ai_gen_dir': f'{PROJECT_ROOT}/datasets/ai_generated',
    'checkpoint_dir': f'{PROJECT_ROOT}/checkpoints',
    'export_dir': f'{PROJECT_ROOT}/exports',

    # --- Model ---
    'backbone': 'tf_efficientnet_b7_ns',  # Noisy Student pretrained - SOTA
    'img_size': 380,                       # EfficientNet-B7 native resolution
    'num_classes': 2,                      # real / fake
    'drop_rate': 0.5,                      # Classifier dropout
    'drop_path_rate': 0.3,                 # Stochastic depth

    # --- Training ---
    'epochs': 30,
    'batch_size': 16,                      # Adjust for your GPU (T4=16, A100=32)
    'accumulation_steps': 2,               # Effective batch = batch_size * accumulation
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'warmup_epochs': 3,
    'min_lr': 1e-7,
    'label_smoothing': 0.05,
    'mixup_alpha': 0.2,
    'cutmix_alpha': 1.0,
    'mixup_prob': 0.5,

    # --- Data ---
    'face_margin': 0.3,                    # 30% margin around face crop
    'num_workers': 2,
    'val_split': 0.15,
    'test_split': 0.10,
    'seed': 42,

    # --- Early Stopping ---
    'patience': 7,
    'min_delta': 1e-4,

    # --- Frequency Analysis ---
    'use_freq_branch': True,               # DCT frequency domain branch
    'freq_weight': 0.3,                    # Weight of frequency features
}

# Reproducibility
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True      # Faster training

print('Config loaded.')
print(f"Effective batch size: {CFG['batch_size'] * CFG['accumulation_steps']}")

---
## Phase 2: Dataset Download & Preparation

### 2A. FaceForensics++ Dataset

In [ ]:
# ============================================================
#  FaceForensics++ Download
# ============================================================
#
#  FF++ requires you to fill out an access form:
#  https://github.com/ondyari/FaceForensics
#
#  Once approved, you get a download script. Run it to get:
#    - original_sequences/youtube/  (1000 real videos)
#    - manipulated_sequences/
#        Deepfakes/            (DF - face swap)
#        Face2Face/            (F2F - facial reenactment)
#        FaceSwap/             (FS - face swap variant)
#        NeuralTextures/       (NT - neural rendering)
#
#  OPTION 1: If you already have FF++ downloaded, upload to Drive
#  OPTION 2: Use the download script below (needs access token)
#  OPTION 3: Use extracted faces dataset from Kaggle (fastest)
# ============================================================

FF_DOWNLOAD_METHOD = 'kaggle'  # 'script' | 'kaggle' | 'manual'

if FF_DOWNLOAD_METHOD == 'kaggle':
    # Kaggle has pre-extracted FF++ face crops - much faster
    # Dataset: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
    # OR the full FF++ faces: https://www.kaggle.com/datasets/ondrejiani/faceforensics-faces
    print('=== Using Kaggle FF++ dataset ===')
    print('Upload your kaggle.json to Colab first:')
    print('  from google.colab import files')
    print('  files.upload()  # upload kaggle.json')
    print()

    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/ 2>/dev/null || echo "Upload kaggle.json first"
    !chmod 600 ~/.kaggle/kaggle.json

    # Download 140k Real and Fake Faces (extracted from FF++)
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces \
        -p {CFG['ff_dataset_dir']} --unzip

elif FF_DOWNLOAD_METHOD == 'script':
    # Official FF++ download script - you need the access token
    print('=== Using official FF++ download script ===')
    print('Get access at: https://github.com/ondyari/FaceForensics')
    !git clone https://github.com/ondyari/FaceForensics.git /tmp/ff_download
    # Uncomment and set your access token:
    # !python /tmp/ff_download/dataset/download.py \
    #     {CFG['ff_dataset_dir']} \
    #     -d all -c c23 -t videos

elif FF_DOWNLOAD_METHOD == 'manual':
    print('=== Manual mode ===')
    print(f"Place FF++ data in: {CFG['ff_dataset_dir']}")
    print('Expected structure:')
    print('  ff++/real/   - real face images')
    print('  ff++/fake/   - fake face images')

### 2B. AI-Generated Images (Gemini & Grok)

In [ ]:
# ============================================================
#  AI-Generated Face Images Collection
# ============================================================
#
#  Strategy: Collect AI-generated face images from multiple sources
#  to make the model robust against latest generators.
#
#  Sources:
#  1. Gemini-generated images (you generate & upload to Drive)
#  2. Grok-generated images (you generate & upload to Drive)
#  3. DALL-E / Midjourney / Stable Diffusion from public datasets
#  4. ThisPersonDoesNotExist (StyleGAN)
# ============================================================

AI_GEN_DIR = CFG['ai_gen_dir']
os.makedirs(f'{AI_GEN_DIR}/gemini', exist_ok=True)
os.makedirs(f'{AI_GEN_DIR}/grok', exist_ok=True)
os.makedirs(f'{AI_GEN_DIR}/stylegan', exist_ok=True)
os.makedirs(f'{AI_GEN_DIR}/diffusion', exist_ok=True)

print('=== AI-Generated Image Collection ===')
print()
print('STEP 1: Generate face images using Gemini & Grok')
print('  - Use prompts like:')
print('    "Generate a realistic photo portrait of a [age] year old [gender]"')
print('    "Professional headshot photo of a person"')
print('    "Candid photo of a person talking"')
print('  - Generate 500-1000 images from each source')
print('  - Save them in your Google Drive at:')
print(f'    Gemini: {AI_GEN_DIR}/gemini/')
print(f'    Grok:   {AI_GEN_DIR}/grok/')
print()
print('STEP 2: Download public AI-generated face datasets')

In [ ]:
# Download public AI-generated datasets

# 1. 1M Fake Faces from Kaggle (StyleGAN generated)
# https://www.kaggle.com/datasets/splcher/animefacedataset or similar
!kaggle datasets download -d ciplab/real-and-fake-face-detection \
    -p {AI_GEN_DIR}/public_dataset --unzip 2>/dev/null || \
    echo 'Kaggle download skipped - upload kaggle.json or download manually'

# 2. ArtiFact dataset - multi-generator AI images
# https://www.kaggle.com/datasets/awsaf49/artifact-dataset
!kaggle datasets download -d awsaf49/artifact-dataset \
    -p {AI_GEN_DIR}/artifact --unzip 2>/dev/null || \
    echo 'ArtiFact download skipped'

print('\nPublic dataset download complete (or skipped).')

### 2C. Face Extraction Pipeline

In [ ]:
# ============================================================
#  MTCNN Face Extractor - Much better than OpenCV DNN
# ============================================================

class FaceExtractor:
    """Extract and align faces using MTCNN."""

    def __init__(self, img_size=380, margin=0.3, device='cuda'):
        self.img_size = img_size
        self.margin = margin
        self.mtcnn = MTCNN(
            image_size=img_size,
            margin=int(img_size * margin),
            select_largest=True,
            post_process=False,  # Return PIL-range [0,255]
            device=device
        )

    def extract(self, img_path):
        """Extract face from image path. Returns numpy array or None."""
        try:
            img = Image.open(img_path).convert('RGB')
            # MTCNN detect + align
            face = self.mtcnn(img)
            if face is not None:
                # face is tensor [3, H, W] in [0, 255]
                face = face.permute(1, 2, 0).numpy().astype(np.uint8)
                return face
            else:
                # Fallback: center crop the image
                img = img.resize((self.img_size, self.img_size), Image.LANCZOS)
                return np.array(img)
        except Exception as e:
            return None

    def extract_from_video(self, video_path, max_frames=32):
        """Extract faces from video frames."""
        cap = cv2.VideoCapture(str(video_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            return []

        indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)
        faces = []

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            face = self.mtcnn(pil_img)
            if face is not None:
                face = face.permute(1, 2, 0).numpy().astype(np.uint8)
                faces.append(face)

        cap.release()
        return faces

face_extractor = FaceExtractor(
    img_size=CFG['img_size'],
    margin=CFG['face_margin'],
    device=str(device)
)
print(f'Face extractor ready (MTCNN, {CFG["img_size"]}px)')

In [ ]:
# ============================================================
#  Build unified dataset: extract faces and organize
# ============================================================

PROCESSED_DIR = f'{PROJECT_ROOT}/datasets/processed'
os.makedirs(f'{PROCESSED_DIR}/real', exist_ok=True)
os.makedirs(f'{PROCESSED_DIR}/fake', exist_ok=True)

def collect_image_paths(base_dir, extensions=('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
    """Recursively collect all image paths."""
    paths = []
    base = Path(base_dir)
    if not base.exists():
        return paths
    for ext in extensions:
        paths.extend(base.rglob(f'*{ext}'))
        paths.extend(base.rglob(f'*{ext.upper()}'))
    return sorted(set(paths))

def process_and_save(image_paths, output_dir, label, extractor, prefix=''):
    """Extract faces and save to output directory."""
    saved = 0
    failed = 0
    for i, img_path in enumerate(tqdm(image_paths, desc=f'{prefix} ({label})')):
        face = extractor.extract(str(img_path))
        if face is not None:
            out_path = f'{output_dir}/{label}/{prefix}_{i:06d}.jpg'
            Image.fromarray(face).save(out_path, quality=95)
            saved += 1
        else:
            failed += 1
    print(f'  {prefix}: saved={saved}, failed={failed}')
    return saved

print('=== Building Unified Dataset ===')
print('This will extract faces from all sources and organize them.\n')

In [ ]:
# ============================================================
#  Process each data source
# ============================================================

stats = {}

# --- FF++ Real ---
ff_real_dirs = [
    f"{CFG['ff_dataset_dir']}/real",
    f"{CFG['ff_dataset_dir']}/real_faces",
    f"{CFG['ff_dataset_dir']}/original_sequences",
]
ff_real_paths = []
for d in ff_real_dirs:
    ff_real_paths.extend(collect_image_paths(d))
if ff_real_paths:
    stats['ff_real'] = process_and_save(ff_real_paths, PROCESSED_DIR, 'real', face_extractor, 'ff_real')

# --- FF++ Fake ---
ff_fake_dirs = [
    f"{CFG['ff_dataset_dir']}/fake",
    f"{CFG['ff_dataset_dir']}/fake_faces",
    f"{CFG['ff_dataset_dir']}/manipulated_sequences",
]
ff_fake_paths = []
for d in ff_fake_dirs:
    ff_fake_paths.extend(collect_image_paths(d))
if ff_fake_paths:
    stats['ff_fake'] = process_and_save(ff_fake_paths, PROCESSED_DIR, 'fake', face_extractor, 'ff_fake')

# --- Gemini Generated (all fake) ---
gemini_paths = collect_image_paths(f'{AI_GEN_DIR}/gemini')
if gemini_paths:
    stats['gemini'] = process_and_save(gemini_paths, PROCESSED_DIR, 'fake', face_extractor, 'gemini')

# --- Grok Generated (all fake) ---
grok_paths = collect_image_paths(f'{AI_GEN_DIR}/grok')
if grok_paths:
    stats['grok'] = process_and_save(grok_paths, PROCESSED_DIR, 'fake', face_extractor, 'grok')

# --- Public AI datasets ---
public_real = collect_image_paths(f'{AI_GEN_DIR}/public_dataset/real_and_fake_face/training_real')
public_real += collect_image_paths(f'{AI_GEN_DIR}/public_dataset/real_and_fake_face/training_fake/../training_real')
if public_real:
    stats['public_real'] = process_and_save(public_real, PROCESSED_DIR, 'real', face_extractor, 'pub_real')

public_fake = collect_image_paths(f'{AI_GEN_DIR}/public_dataset/real_and_fake_face/training_fake')
if public_fake:
    stats['public_fake'] = process_and_save(public_fake, PROCESSED_DIR, 'fake', face_extractor, 'pub_fake')

# --- Summary ---
total_real = len(list(Path(f'{PROCESSED_DIR}/real').glob('*.jpg')))
total_fake = len(list(Path(f'{PROCESSED_DIR}/fake').glob('*.jpg')))
print(f'\n{"="*50}')
print(f'TOTAL REAL: {total_real:,}')
print(f'TOTAL FAKE: {total_fake:,}')
print(f'TOTAL:      {total_real + total_fake:,}')
print(f'Ratio:      1:{total_fake/max(total_real,1):.2f} (real:fake)')
print(f'{"="*50}')

---
## Phase 3: Data Pipeline

In [ ]:
# ============================================================
#  Augmentation Pipeline - Aggressive but realistic
# ============================================================

def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1),
        ], p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1),
            A.GaussNoise(var_limit=(10, 50), p=1),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1),
        ], p=0.3),
        A.OneOf([
            A.ImageCompression(quality_lower=60, quality_upper=100, p=1),
            A.Downscale(scale_min=0.5, scale_max=0.9, p=1),
        ], p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.3),
        A.CoarseDropout(max_holes=4, max_height=30, max_width=30, p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

print('Augmentation pipelines defined.')

In [ ]:
# ============================================================
#  DCT Frequency Domain Feature Extraction
# ============================================================
#  GAN/diffusion artifacts leave traces in frequency domain
#  that are invisible to the naked eye but detectable by models.
# ============================================================

def extract_dct_features(img_np, block_size=8):
    """Extract DCT frequency features from image.
    Deepfakes leave spectral fingerprints in high-frequency components.
    """
    if len(img_np.shape) == 3:
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_np

    h, w = gray.shape
    # Crop to multiple of block_size
    h = h - h % block_size
    w = w - w % block_size
    gray = gray[:h, :w].astype(np.float32)

    # Full image DCT
    dct = cv2.dct(gray / 255.0)

    # Log-scaled magnitude spectrum
    dct_log = np.log1p(np.abs(dct))

    # Resize to fixed size for the frequency branch
    dct_resized = cv2.resize(dct_log, (64, 64))

    # Normalize
    dct_resized = (dct_resized - dct_resized.mean()) / (dct_resized.std() + 1e-8)

    return dct_resized

print('DCT feature extraction ready.')

In [ ]:
# ============================================================
#  Dataset Class
# ============================================================

class DeepfakeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None, use_freq=True):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.use_freq = use_freq

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Load image
        img = cv2.imread(str(img_path))
        if img is None:
            # Return a black image on failure
            img = np.zeros((CFG['img_size'], CFG['img_size'], 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # DCT features (before augmentation to preserve artifacts)
        freq_features = None
        if self.use_freq:
            freq_features = extract_dct_features(img)
            freq_features = torch.FloatTensor(freq_features).unsqueeze(0)  # [1, 64, 64]

        # Augmentation
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']

        if freq_features is not None:
            return img, freq_features, label
        return img, label

print('Dataset class defined.')

In [ ]:
# ============================================================
#  Build Train / Val / Test splits
# ============================================================

from sklearn.model_selection import train_test_split

# Collect all processed images
real_images = sorted(Path(f'{PROCESSED_DIR}/real').glob('*.jpg'))
fake_images = sorted(Path(f'{PROCESSED_DIR}/fake').glob('*.jpg'))

all_paths = list(real_images) + list(fake_images)
all_labels = [0] * len(real_images) + [1] * len(fake_images)  # 0=real, 1=fake

print(f'Total images: {len(all_paths):,}')
print(f'  Real: {len(real_images):,}')
print(f'  Fake: {len(fake_images):,}')

# Split: train / (val + test)
train_paths, valtest_paths, train_labels, valtest_labels = train_test_split(
    all_paths, all_labels,
    test_size=CFG['val_split'] + CFG['test_split'],
    stratify=all_labels,
    random_state=CFG['seed']
)

# Split val+test into val / test
relative_test = CFG['test_split'] / (CFG['val_split'] + CFG['test_split'])
val_paths, test_paths, val_labels, test_labels = train_test_split(
    valtest_paths, valtest_labels,
    test_size=relative_test,
    stratify=valtest_labels,
    random_state=CFG['seed']
)

print(f'\nSplit sizes:')
print(f'  Train: {len(train_paths):,} (real={sum(1 for l in train_labels if l==0):,}, fake={sum(1 for l in train_labels if l==1):,})')
print(f'  Val:   {len(val_paths):,} (real={sum(1 for l in val_labels if l==0):,}, fake={sum(1 for l in val_labels if l==1):,})')
print(f'  Test:  {len(test_paths):,} (real={sum(1 for l in test_labels if l==0):,}, fake={sum(1 for l in test_labels if l==1):,})')

In [ ]:
# ============================================================
#  Create DataLoaders with class balancing
# ============================================================

use_freq = CFG['use_freq_branch']

train_dataset = DeepfakeDataset(
    train_paths, train_labels,
    transform=get_train_transforms(CFG['img_size']),
    use_freq=use_freq
)
val_dataset = DeepfakeDataset(
    val_paths, val_labels,
    transform=get_val_transforms(CFG['img_size']),
    use_freq=use_freq
)
test_dataset = DeepfakeDataset(
    test_paths, test_labels,
    transform=get_val_transforms(CFG['img_size']),
    use_freq=use_freq
)

# Weighted sampler to handle class imbalance
class_counts = Counter(train_labels)
class_weights = {c: 1.0 / count for c, count in class_counts.items()}
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=CFG['batch_size'],
    sampler=sampler, num_workers=CFG['num_workers'],
    pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG['batch_size'] * 2,
    shuffle=False, num_workers=CFG['num_workers'],
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=CFG['batch_size'] * 2,
    shuffle=False, num_workers=CFG['num_workers'],
    pin_memory=True
)

print(f'DataLoaders ready.')
print(f'  Train batches: {len(train_loader)}')
print(f'  Val batches:   {len(val_loader)}')
print(f'  Test batches:  {len(test_loader)}')

---
## Phase 4: Model Architecture

**Design:** EfficientNet-B7 (Noisy Student) backbone + Frequency Analysis Branch + Multi-Head Attention Pooling

This combines:
1. **Spatial features** from EfficientNet-B7 (best ImageNet pretrained backbone)
2. **Frequency features** from DCT analysis (catches GAN/diffusion spectral artifacts)
3. **Attention pooling** to focus on manipulated regions

In [ ]:
# ============================================================
#  Attention Pooling Module
# ============================================================

class MultiHeadAttentionPooling(nn.Module):
    """Attention-weighted global pooling.
    Learns to focus on manipulated facial regions."""

    def __init__(self, in_features, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = in_features // num_heads

        self.query = nn.Linear(in_features, in_features)
        self.key = nn.Linear(in_features, in_features)
        self.value = nn.Linear(in_features, in_features)
        self.out_proj = nn.Linear(in_features, in_features)
        self.norm = nn.LayerNorm(in_features)

    def forward(self, x):
        # x: [B, C, H, W] -> [B, H*W, C]
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)  # [B, N, C]

        # Global token as query
        global_token = x_flat.mean(dim=1, keepdim=True)  # [B, 1, C]

        q = self.query(global_token)  # [B, 1, C]
        k = self.key(x_flat)          # [B, N, C]
        v = self.value(x_flat)        # [B, N, C]

        # Multi-head reshape
        q = q.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention
        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)

        out = (attn @ v).transpose(1, 2).reshape(B, 1, C)
        out = self.out_proj(out).squeeze(1)
        out = self.norm(out)

        return out  # [B, C]

print('Attention pooling module defined.')

In [ ]:
# ============================================================
#  Frequency Analysis Branch
# ============================================================

class FrequencyBranch(nn.Module):
    """Lightweight CNN to process DCT frequency features."""

    def __init__(self, out_features=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(256, out_features)

    def forward(self, x):
        # x: [B, 1, 64, 64]
        x = self.conv(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x  # [B, out_features]

print('Frequency branch defined.')

In [ ]:
# ============================================================
#  BEAST MODEL: EfficientNet-B7 + Freq + Attention
# ============================================================

class DeepfakeBeastDetector(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.use_freq = cfg['use_freq_branch']

        # --- Backbone: EfficientNet-B7 Noisy Student ---
        self.backbone = timm.create_model(
            cfg['backbone'],
            pretrained=True,
            num_classes=0,  # Remove classifier
            drop_rate=cfg['drop_rate'],
            drop_path_rate=cfg['drop_path_rate'],
        )
        backbone_features = self.backbone.num_features  # 2560 for B7

        # --- Attention Pooling ---
        # Replace global avg pool with attention-weighted pooling
        self.attn_pool = MultiHeadAttentionPooling(backbone_features, num_heads=8)

        # --- Frequency Branch ---
        freq_features = 0
        if self.use_freq:
            self.freq_branch = FrequencyBranch(out_features=256)
            freq_features = 256

        # --- Classifier Head ---
        total_features = backbone_features + freq_features
        self.classifier = nn.Sequential(
            nn.Linear(total_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(cfg['drop_rate']),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(cfg['drop_rate'] * 0.5),
            nn.Linear(512, cfg['num_classes']),
        )

        # Initialize classifier weights
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_features(self, x):
        """Extract spatial features with attention pooling."""
        # Get feature map before global pool
        features = self.backbone.forward_features(x)  # [B, C, H, W]
        # Attention-weighted pooling
        pooled = self.attn_pool(features)  # [B, C]
        return pooled

    def forward(self, x, freq_input=None):
        # Spatial features
        spatial = self.forward_features(x)  # [B, 2560]

        # Frequency features
        if self.use_freq and freq_input is not None:
            freq = self.freq_branch(freq_input)  # [B, 256]
            combined = torch.cat([spatial, freq], dim=1)  # [B, 2816]
        else:
            combined = spatial

        logits = self.classifier(combined)  # [B, 2]
        return logits

# Build model
model = DeepfakeBeastDetector(CFG).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: DeepfakeBeastDetector')
print(f'  Backbone: {CFG["backbone"]}')
print(f'  Total params:     {total_params:,}')
print(f'  Trainable params: {trainable_params:,}')
print(f'  Freq branch:      {CFG["use_freq_branch"]}')

---
## Phase 5: Training Engine

In [ ]:
# ============================================================
#  Mixup / CutMix for regularization
# ============================================================

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    _, _, H, W = x.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print('Mixup/CutMix ready.')

In [ ]:
# ============================================================
#  Cosine Annealing with Warmup Scheduler
# ============================================================

class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-7):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.step_count = 0

    def step(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            # Linear warmup
            scale = self.step_count / self.warmup_steps
        else:
            # Cosine annealing
            progress = (self.step_count - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            scale = 0.5 * (1 + np.cos(np.pi * progress))

        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = max(self.min_lr, base_lr * scale)

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

print('Scheduler ready.')

In [ ]:
# ============================================================
#  Setup optimizer, loss, scheduler, scaler
# ============================================================

# Layer-wise learning rate decay - backbone gets lower LR
backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if 'backbone' in name:
        backbone_params.append(param)
    else:
        head_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr'] * 0.1},   # Backbone: lower LR
    {'params': head_params, 'lr': CFG['lr']},              # Head: full LR
], weight_decay=CFG['weight_decay'])

# Loss with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])

# Scheduler
total_steps = len(train_loader) * CFG['epochs'] // CFG['accumulation_steps']
warmup_steps = len(train_loader) * CFG['warmup_epochs'] // CFG['accumulation_steps']
scheduler = CosineWarmupScheduler(optimizer, warmup_steps, total_steps, CFG['min_lr'])

# Mixed precision scaler
scaler = GradScaler()

print(f'Optimizer: AdamW (backbone_lr={CFG["lr"]*0.1:.1e}, head_lr={CFG["lr"]:.1e})')
print(f'Scheduler: CosineWarmup (warmup={warmup_steps}, total={total_steps})')
print(f'Loss: CrossEntropy (label_smoothing={CFG["label_smoothing"]})')
print(f'Mixed Precision: FP16')

In [ ]:
# ============================================================
#  Training Loop
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, scheduler, scaler, cfg, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'Train E{epoch+1}')
    for step, batch in enumerate(pbar):
        if cfg['use_freq_branch']:
            images, freq, labels = batch
            freq = freq.to(device, non_blocking=True)
        else:
            images, labels = batch
            freq = None

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Mixup / CutMix
        use_mix = np.random.random() < cfg['mixup_prob'] and epoch >= cfg['warmup_epochs']
        if use_mix:
            if np.random.random() < 0.5:
                images, labels_a, labels_b, lam = mixup_data(images, labels, cfg['mixup_alpha'])
            else:
                images, labels_a, labels_b, lam = cutmix_data(images, labels, cfg['cutmix_alpha'])

        # Forward pass with mixed precision
        with autocast():
            logits = model(images, freq)
            if use_mix:
                loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
            else:
                loss = criterion(logits, labels)
            loss = loss / cfg['accumulation_steps']

        # Backward
        scaler.scale(loss).backward()

        if (step + 1) % cfg['accumulation_steps'] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        # Stats
        running_loss += loss.item() * cfg['accumulation_steps']
        if not use_mix:
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        pbar.set_postfix({
            'loss': f'{running_loss/(step+1):.4f}',
            'acc': f'{correct/max(total,1)*100:.1f}%',
            'lr': f'{scheduler.get_lr():.2e}'
        })

    return running_loss / len(loader), correct / max(total, 1)

print('Training loop defined.')

In [ ]:
# ============================================================
#  Validation Loop
# ============================================================

@torch.no_grad()
def validate(model, loader, criterion, cfg):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_probs = []
    all_labels = []

    for batch in tqdm(loader, desc='Validating'):
        if cfg['use_freq_branch']:
            images, freq, labels = batch
            freq = freq.to(device, non_blocking=True)
        else:
            images, labels = batch
            freq = None

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast():
            logits = model(images, freq)
            loss = criterion(logits, labels)

        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)

        running_loss += loss.item()
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # Fake probability
        all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    auc = roc_auc_score(all_labels, all_probs)

    metrics = {
        'loss': running_loss / len(loader),
        'accuracy': acc,
        'f1': f1,
        'auc_roc': auc,
        'preds': all_preds,
        'probs': all_probs,
        'labels': all_labels,
    }
    return metrics

print('Validation loop defined.')

In [ ]:
# ============================================================
#  MAIN TRAINING LOOP
# ============================================================

best_auc = 0.0
best_f1 = 0.0
best_epoch = 0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': [], 'lr': []}

print('=' * 60)
print('  TRAINING STARTED')
print('=' * 60)

for epoch in range(CFG['epochs']):
    start_time = time.time()

    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scheduler, scaler, CFG, epoch
    )

    # Validate
    val_metrics = validate(model, val_loader, criterion, CFG)

    elapsed = time.time() - start_time

    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_auc'].append(val_metrics['auc_roc'])
    history['lr'].append(scheduler.get_lr())

    print(f'\nEpoch {epoch+1}/{CFG["epochs"]} ({elapsed:.0f}s)')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%')
    print(f'  Val Loss:   {val_metrics["loss"]:.4f} | Val Acc:   {val_metrics["accuracy"]*100:.2f}%')
    print(f'  Val F1:     {val_metrics["f1"]:.4f} | Val AUC:   {val_metrics["auc_roc"]:.4f}')
    print(f'  LR: {scheduler.get_lr():.2e}')

    # Save best model (by AUC-ROC)
    improved = False
    if val_metrics['auc_roc'] > best_auc + CFG['min_delta']:
        best_auc = val_metrics['auc_roc']
        best_f1 = val_metrics['f1']
        best_epoch = epoch + 1
        patience_counter = 0
        improved = True

        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_auc': best_auc,
            'best_f1': best_f1,
            'config': CFG,
            'history': history,
        }
        torch.save(checkpoint, f'{CFG["checkpoint_dir"]}/best_model.pt')
        print(f'  >>> NEW BEST MODEL SAVED (AUC={best_auc:.4f}, F1={best_f1:.4f})')
    else:
        patience_counter += 1
        print(f'  No improvement ({patience_counter}/{CFG["patience"]})')

    # Save periodic checkpoint
    if (epoch + 1) % 5 == 0:
        torch.save(checkpoint if improved else {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'config': CFG,
        }, f'{CFG["checkpoint_dir"]}/checkpoint_epoch{epoch+1}.pt')

    # Early stopping
    if patience_counter >= CFG['patience']:
        print(f'\nEarly stopping at epoch {epoch+1}!')
        break

print(f'\n{"="*60}')
print(f'TRAINING COMPLETE')
print(f'Best Epoch: {best_epoch} | AUC: {best_auc:.4f} | F1: {best_f1:.4f}')
print(f'{"="*60}')

---
## Phase 6: Comprehensive Evaluation

In [ ]:
# ============================================================
#  Load best model and evaluate on test set
# ============================================================

# Load best checkpoint
checkpoint = torch.load(f'{CFG["checkpoint_dir"]}/best_model.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]} (AUC={checkpoint["best_auc"]:.4f})')

# Evaluate on test set
test_metrics = validate(model, test_loader, criterion, CFG)

print(f'\n{"="*60}')
print(f'  TEST SET RESULTS')
print(f'{"="*60}')
print(f'  Accuracy:  {test_metrics["accuracy"]*100:.2f}%')
print(f'  F1 Score:  {test_metrics["f1"]:.4f}')
print(f'  AUC-ROC:   {test_metrics["auc_roc"]:.4f}')
print(f'{"="*60}')

In [ ]:
# ============================================================
#  Detailed Classification Report
# ============================================================

print('\nClassification Report:')
print('=' * 50)
print(classification_report(
    test_metrics['labels'],
    test_metrics['preds'],
    target_names=['REAL', 'FAKE'],
    digits=4
))

# Per-class metrics
from sklearn.metrics import precision_score, recall_score
for cls, name in [(0, 'REAL'), (1, 'FAKE')]:
    mask = test_metrics['labels'] == cls
    cls_acc = (test_metrics['preds'][mask] == cls).mean()
    print(f'{name}: accuracy={cls_acc*100:.2f}%, samples={mask.sum()}')

In [ ]:
# ============================================================
#  Visualization: Confusion Matrix, ROC, PR Curve, Loss Curve
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Deepfake Beast Model - Evaluation Dashboard', fontsize=16, fontweight='bold')

# 1. Confusion Matrix
cm = confusion_matrix(test_metrics['labels'], test_metrics['preds'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
axes[0, 0].set_title('Confusion Matrix')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('Actual')

# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(test_metrics['labels'], test_metrics['probs'])
auc_score = roc_auc_score(test_metrics['labels'], test_metrics['probs'])
axes[0, 1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {auc_score:.4f}')
axes[0, 1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
axes[0, 1].set_title('ROC Curve')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].legend(fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# 3. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(test_metrics['labels'], test_metrics['probs'])
ap = average_precision_score(test_metrics['labels'], test_metrics['probs'])
axes[0, 2].plot(recall, precision, 'g-', linewidth=2, label=f'AP = {ap:.4f}')
axes[0, 2].set_title('Precision-Recall Curve')
axes[0, 2].set_xlabel('Recall')
axes[0, 2].set_ylabel('Precision')
axes[0, 2].legend(fontsize=12)
axes[0, 2].grid(True, alpha=0.3)

# 4. Training Loss Curve
axes[1, 0].plot(history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[1, 0].plot(history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[1, 0].set_title('Loss Curves')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 5. Accuracy & F1 Curve
axes[1, 1].plot(history['val_acc'], 'b-', label='Accuracy', linewidth=2)
axes[1, 1].plot(history['val_f1'], 'g-', label='F1 Score', linewidth=2)
axes[1, 1].plot(history['val_auc'], 'r-', label='AUC-ROC', linewidth=2)
axes[1, 1].set_title('Validation Metrics')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. Score Distribution
real_probs = test_metrics['probs'][test_metrics['labels'] == 0]
fake_probs = test_metrics['probs'][test_metrics['labels'] == 1]
axes[1, 2].hist(real_probs, bins=50, alpha=0.6, color='green', label='Real', density=True)
axes[1, 2].hist(fake_probs, bins=50, alpha=0.6, color='red', label='Fake', density=True)
axes[1, 2].axvline(x=0.5, color='black', linestyle='--', alpha=0.5)
axes[1, 2].set_title('Score Distribution')
axes[1, 2].set_xlabel('Fake Probability')
axes[1, 2].set_ylabel('Density')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG["project_root"]}/evaluation_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Dashboard saved to {CFG["project_root"]}/evaluation_dashboard.png')

In [ ]:
# ============================================================
#  Find Optimal Threshold (maximizes F1)
# ============================================================

from sklearn.metrics import f1_score as compute_f1

thresholds = np.arange(0.1, 0.95, 0.01)
f1_scores = []
for t in thresholds:
    preds_t = (test_metrics['probs'] >= t).astype(int)
    f1_scores.append(compute_f1(test_metrics['labels'], preds_t, average='weighted'))

optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

# Recalculate metrics at optimal threshold
optimal_preds = (test_metrics['probs'] >= optimal_threshold).astype(int)
optimal_acc = accuracy_score(test_metrics['labels'], optimal_preds)

print(f'Optimal Threshold: {optimal_threshold:.2f}')
print(f'  Accuracy at optimal: {optimal_acc*100:.2f}%')
print(f'  F1 at optimal:       {optimal_f1:.4f}')
print(f'  AUC-ROC:             {test_metrics["auc_roc"]:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, 'b-', linewidth=2)
plt.axvline(x=optimal_threshold, color='r', linestyle='--', label=f'Optimal={optimal_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Phase 7: Export Model for Production

In [ ]:
# ============================================================
#  Export 1: PyTorch model for ML service
# ============================================================

export_dir = CFG['export_dir']

# Save full model for the ML service
export_checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': CFG,
    'optimal_threshold': float(optimal_threshold),
    'metrics': {
        'accuracy': float(optimal_acc),
        'f1_score': float(optimal_f1),
        'auc_roc': float(test_metrics['auc_roc']),
    },
    'class_names': ['real', 'fake'],
    'architecture': 'DeepfakeBeastDetector',
    'backbone': CFG['backbone'],
    'img_size': CFG['img_size'],
    'use_freq_branch': CFG['use_freq_branch'],
}
torch.save(export_checkpoint, f'{export_dir}/deepfake_beast_model.pt')
print(f'Saved: {export_dir}/deepfake_beast_model.pt')

# Save config as JSON
config_export = {
    'architecture': 'DeepfakeBeastDetector',
    'backbone': CFG['backbone'],
    'img_size': CFG['img_size'],
    'num_classes': CFG['num_classes'],
    'use_freq_branch': CFG['use_freq_branch'],
    'optimal_threshold': float(optimal_threshold),
    'normalization': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225],
    },
    'class_names': ['real', 'fake'],
    'test_metrics': {
        'accuracy': float(optimal_acc),
        'f1_score': float(optimal_f1),
        'auc_roc': float(test_metrics['auc_roc']),
    },
    'training_data': {
        'datasets': ['FaceForensics++', 'Gemini-generated', 'Grok-generated', 'Public-AI-faces'],
        'train_samples': len(train_paths),
        'val_samples': len(val_paths),
        'test_samples': len(test_paths),
    }
}
with open(f'{export_dir}/model_config.json', 'w') as f:
    json.dump(config_export, f, indent=2)
print(f'Saved: {export_dir}/model_config.json')

In [ ]:
# ============================================================
#  Export 2: ONNX for optimized inference
# ============================================================

model.eval()
dummy_img = torch.randn(1, 3, CFG['img_size'], CFG['img_size']).to(device)
dummy_freq = torch.randn(1, 1, 64, 64).to(device) if CFG['use_freq_branch'] else None

input_names = ['image']
dynamic_axes = {'image': {0: 'batch'}, 'output': {0: 'batch'}}
inputs = (dummy_img,)

if CFG['use_freq_branch']:
    input_names.append('freq')
    dynamic_axes['freq'] = {0: 'batch'}
    inputs = (dummy_img, dummy_freq)

onnx_path = f'{export_dir}/deepfake_beast_model.onnx'
torch.onnx.export(
    model, inputs, onnx_path,
    input_names=input_names,
    output_names=['output'],
    dynamic_axes=dynamic_axes,
    opset_version=17,
)
print(f'Saved ONNX: {onnx_path}')

# Model size
pt_size = os.path.getsize(f'{export_dir}/deepfake_beast_model.pt') / 1e6
onnx_size = os.path.getsize(onnx_path) / 1e6
print(f'\nModel sizes:')
print(f'  PyTorch: {pt_size:.1f} MB')
print(f'  ONNX:    {onnx_size:.1f} MB')

In [ ]:
# ============================================================
#  Export 3: TorchScript for deployment
# ============================================================

model.eval()
try:
    if CFG['use_freq_branch']:
        scripted = torch.jit.trace(model, (dummy_img, dummy_freq))
    else:
        scripted = torch.jit.trace(model, (dummy_img,))
    scripted.save(f'{export_dir}/deepfake_beast_model_scripted.pt')
    print(f'Saved TorchScript: {export_dir}/deepfake_beast_model_scripted.pt')
except Exception as e:
    print(f'TorchScript export failed (non-critical): {e}')

---
## Phase 8: Integration Helper

This generates the inference wrapper that plugs into your existing ML service.

In [ ]:
# ============================================================
#  Generate inference wrapper for your ML service
# ============================================================

inference_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import cv2
import numpy as np
from PIL import Image
from facenet_pytorch import MTCNN
import json


class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, in_features, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = in_features // num_heads
        self.query = nn.Linear(in_features, in_features)
        self.key = nn.Linear(in_features, in_features)
        self.value = nn.Linear(in_features, in_features)
        self.out_proj = nn.Linear(in_features, in_features)
        self.norm = nn.LayerNorm(in_features)

    def forward(self, x):
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)
        global_token = x_flat.mean(dim=1, keepdim=True)
        q = self.query(global_token)
        k = self.key(x_flat)
        v = self.value(x_flat)
        q = q.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        scale = self.head_dim ** -0.5
        attn = F.softmax((q @ k.transpose(-2, -1)) * scale, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, 1, C)
        return self.norm(self.out_proj(out).squeeze(1))


class FrequencyBranch(nn.Module):
    def __init__(self, out_features=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(256, out_features)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))


class DeepfakeBeastDetector(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.use_freq = cfg.get("use_freq_branch", True)
        self.backbone = timm.create_model(
            cfg["backbone"], pretrained=False, num_classes=0,
            drop_rate=cfg.get("drop_rate", 0.5),
            drop_path_rate=cfg.get("drop_path_rate", 0.3),
        )
        backbone_features = self.backbone.num_features
        self.attn_pool = MultiHeadAttentionPooling(backbone_features, num_heads=8)
        freq_features = 0
        if self.use_freq:
            self.freq_branch = FrequencyBranch(out_features=256)
            freq_features = 256
        total_features = backbone_features + freq_features
        self.classifier = nn.Sequential(
            nn.Linear(total_features, 1024), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Dropout(cfg.get("drop_rate", 0.5)),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(cfg.get("drop_rate", 0.5) * 0.5),
            nn.Linear(512, cfg.get("num_classes", 2)),
        )

    def forward_features(self, x):
        return self.attn_pool(self.backbone.forward_features(x))

    def forward(self, x, freq_input=None):
        spatial = self.forward_features(x)
        if self.use_freq and freq_input is not None:
            combined = torch.cat([spatial, self.freq_branch(freq_input)], dim=1)
        else:
            combined = spatial
        return self.classifier(combined)


class BeastModelInference:
    """Inference wrapper for the Deepfake Beast model.
    Drop-in replacement for your existing ML service."""

    def __init__(self, model_path, config_path=None, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")

        # Load checkpoint
        checkpoint = torch.load(model_path, map_location=self.device)
        cfg = checkpoint["config"]
        self.img_size = cfg["img_size"]
        self.use_freq = cfg.get("use_freq_branch", True)
        self.threshold = checkpoint.get("optimal_threshold", 0.5)
        self.mean = np.array([0.485, 0.456, 0.406])
        self.std = np.array([0.229, 0.224, 0.225])

        # Build and load model
        self.model = DeepfakeBeastDetector(cfg).to(self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

        # Face detector
        self.mtcnn = MTCNN(
            image_size=self.img_size,
            margin=int(self.img_size * 0.3),
            select_largest=True,
            post_process=False,
            device=self.device
        )

    def extract_dct(self, img_np):
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY) if len(img_np.shape) == 3 else img_np
        h, w = gray.shape
        h, w = h - h % 8, w - w % 8
        dct = cv2.dct(gray[:h, :w].astype(np.float32) / 255.0)
        dct_log = np.log1p(np.abs(dct))
        dct_resized = cv2.resize(dct_log, (64, 64))
        dct_resized = (dct_resized - dct_resized.mean()) / (dct_resized.std() + 1e-8)
        return torch.FloatTensor(dct_resized).unsqueeze(0).unsqueeze(0).to(self.device)

    def preprocess(self, img_path):
        img = Image.open(img_path).convert("RGB")
        face = self.mtcnn(img)
        if face is not None:
            face_np = face.permute(1, 2, 0).numpy().astype(np.uint8)
        else:
            face_np = np.array(img.resize((self.img_size, self.img_size)))

        # Normalize
        tensor = torch.FloatTensor(face_np).permute(2, 0, 1) / 255.0
        for c in range(3):
            tensor[c] = (tensor[c] - self.mean[c]) / self.std[c]

        freq = self.extract_dct(face_np) if self.use_freq else None
        return tensor.unsqueeze(0).to(self.device), freq, face is not None

    @torch.no_grad()
    def predict(self, img_path):
        tensor, freq, face_detected = self.preprocess(img_path)
        logits = self.model(tensor, freq)
        probs = F.softmax(logits, dim=1)[0]
        fake_prob = probs[1].item()
        return {
            "fake_probability": fake_prob,
            "real_probability": probs[0].item(),
            "is_fake": fake_prob >= self.threshold,
            "confidence": abs(fake_prob - 0.5) * 2,
            "face_detected": face_detected,
            "threshold": self.threshold,
        }

    @torch.no_grad()
    def predict_batch(self, img_paths):
        return [self.predict(p) for p in img_paths]
'''

with open(f'{export_dir}/beast_inference.py', 'w') as f:
    f.write(inference_code)

print(f'Saved inference wrapper: {export_dir}/beast_inference.py')
print()
print('Usage in your ML service:')
print('  from beast_inference import BeastModelInference')
print('  model = BeastModelInference("deepfake_beast_model.pt")')
print('  result = model.predict("image.jpg")')

---
## Phase 9: Final Summary & Next Steps

In [ ]:
# ============================================================
#  Print final summary
# ============================================================

print('=' * 60)
print('  DEEPFAKE BEAST MODEL - TRAINING COMPLETE')
print('=' * 60)
print()
print(f'Architecture: EfficientNet-B7 (Noisy Student) + Freq Branch + Attention Pooling')
print(f'Image Size:   {CFG["img_size"]}x{CFG["img_size"]}')
print(f'Parameters:   {total_params:,}')
print()
print(f'--- Test Results ---')
print(f'Accuracy:     {optimal_acc*100:.2f}%')
print(f'F1 Score:     {optimal_f1:.4f}')
print(f'AUC-ROC:      {test_metrics["auc_roc"]:.4f}')
print(f'Threshold:    {optimal_threshold:.2f}')
print()
print(f'--- Exported Files ---')
print(f'  {export_dir}/deepfake_beast_model.pt        (PyTorch checkpoint)')
print(f'  {export_dir}/deepfake_beast_model.onnx      (ONNX optimized)')
print(f'  {export_dir}/model_config.json              (Config + metrics)')
print(f'  {export_dir}/beast_inference.py             (Inference wrapper)')
print()
print('--- Integration Steps ---')
print('1. Copy deepfake_beast_model.pt to your ml-service/ directory')
print('2. Copy beast_inference.py to ml-service/')
print('3. Update app.py to use BeastModelInference instead of SigLIP pipeline')
print('4. Update requirements.txt: add timm, facenet-pytorch')
print('5. Update Dockerfile to install new dependencies')
print('6. Test with docker-compose up')